# NB_hospital_daily_summary

Builds `adb_healthcare.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from `silver.lab_results`, enriched with hospital names from `silver.hospitals` and day attributes from `silver.calendar`, per the STTM workbook `sttm_silver_to_gold.xlsx`. Full rebuild via CREATE OR REPLACE TABLE, with LAG-based day-over-day test volume trends.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date AS summary_date,
    COUNT(*) AS total_tests,
    COUNT(DISTINCT patient_id) AS unique_patients_tested,
    COUNT(DISTINCT doctor_id) AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM adb_healthcare.silver.lab_results
  GROUP BY hospital_id, test_date
),
trended AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  t.hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT) AS total_tests,
  CAST(t.unique_patients_tested AS INT) AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT) AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT) AS abnormal_test_count,
  CAST(t.critical_test_count AS INT) AS critical_test_count,
  CAST(t.abnormal_test_count * 100.0 / t.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(t.critical_test_count * 100.0 / t.total_tests AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT) AS prev_day_total_tests,
  CAST(
    CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
         ELSE (t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests END
    AS DECIMAL(6,2)) AS test_volume_change_pct
FROM trended t
LEFT JOIN adb_healthcare.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_healthcare.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%sql
WITH expected AS (
  SELECT * FROM VALUES
    (1,'hospital_id','int'),(2,'hospital_name','string'),(3,'summary_date','date'),(4,'day_name','string'),
    (5,'is_weekend','boolean'),(6,'total_tests','int'),(7,'unique_patients_tested','int'),
    (8,'active_doctor_count','int'),(9,'abnormal_test_count','int'),(10,'critical_test_count','int'),
    (11,'abnormal_rate_pct','decimal(5,2)'),(12,'critical_rate_pct','decimal(5,2)'),
    (13,'prev_day_total_tests','int'),(14,'test_volume_change_pct','decimal(6,2)')
  AS t(position, column_name, expected_type)
),
actual AS (
  SELECT ordinal_position + 1 AS position, column_name,
         lower(CASE WHEN data_type = 'DECIMAL' THEN concat('decimal(', numeric_precision, ',', numeric_scale, ')') ELSE data_type END) AS actual_type
  FROM adb_healthcare.information_schema.columns
  WHERE table_schema = 'gold' AND table_name = 'hospital_daily_summary'
)
SELECT
  COALESCE(e.position, a.position) AS position,
  COALESCE(e.column_name, a.column_name) AS column_name,
  e.expected_type,
  a.actual_type,
  CASE WHEN e.column_name IS NULL THEN 'UNEXPECTED COLUMN'
       WHEN a.column_name IS NULL THEN 'MISSING'
       WHEN e.expected_type <> a.actual_type THEN 'TYPE MISMATCH'
       WHEN e.position <> a.position THEN 'ORDER MISMATCH'
       ELSE 'OK' END AS status
FROM expected e
FULL OUTER JOIN actual a ON e.column_name = a.column_name
ORDER BY position;

In [0]:
%sql
SELECT
  COUNT(*) AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date) AS distinct_grain_keys,
  (SELECT COUNT(DISTINCT hospital_id, test_date) FROM adb_healthcare.silver.lab_results) AS silver_hospital_days,
  COUNT(DISTINCT hospital_id) AS hospitals,
  MIN(summary_date) AS first_date,
  MAX(summary_date) AS last_date,
  SUM(total_tests) AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_healthcare.silver.lab_results) AS silver_total_tests,
  SUM(abnormal_test_count) AS gold_abnormal,
  SUM(critical_test_count) AS gold_critical,
  COUNT_IF(hospital_id IS NULL) AS null_hospital_id,
  COUNT_IF(hospital_name IS NULL) AS missing_hospital_name,
  COUNT_IF(day_name IS NULL OR is_weekend IS NULL) AS missing_calendar_enrichment,
  COUNT_IF(day_name <> date_format(summary_date, 'EEEE')) AS day_name_mismatches,
  COUNT_IF(is_weekend <> (dayofweek(summary_date) IN (1, 7))) AS is_weekend_mismatches,
  COUNT_IF(unique_patients_tested > total_tests OR active_doctor_count > total_tests) AS distinct_count_violations
FROM adb_healthcare.gold.hospital_daily_summary;

In [0]:
%sql
WITH chk AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY summary_date) AS rn,
    LAG(summary_date) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_date,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS expected_prev,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS expected_abn_rate,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS expected_crit_rate
  FROM adb_healthcare.gold.hospital_daily_summary
)
SELECT
  COUNT_IF(abnormal_rate_pct <> expected_abn_rate) AS abnormal_rate_mismatches,
  COUNT_IF(critical_rate_pct <> expected_crit_rate) AS critical_rate_mismatches,
  COUNT_IF(rn = 1 AND prev_day_total_tests IS NOT NULL) AS first_day_prev_not_null,
  COUNT_IF(rn > 1 AND prev_day_total_tests IS DISTINCT FROM expected_prev) AS prev_mismatches,
  COUNT_IF(test_volume_change_pct IS DISTINCT FROM
    CAST(CASE WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
              ELSE (total_tests - prev_day_total_tests) * 100.0 / prev_day_total_tests END AS DECIMAL(6,2))) AS change_mismatches,
  COUNT_IF(rn = 1) AS first_day_rows,
  COUNT_IF(prev_day_total_tests IS NULL) AS null_prev_rows,
  COUNT_IF(test_volume_change_pct IS NULL) AS null_change_rows,
  COUNT_IF(rn > 1 AND datediff(summary_date, prev_date) > 1) AS rows_after_day_gap,
  ROUND(AVG(CASE WHEN rn > 1 THEN datediff(summary_date, prev_date) END), 1) AS avg_days_between_activity,
  MIN(test_volume_change_pct) AS min_change_pct,
  MAX(test_volume_change_pct) AS max_change_pct
FROM chk;

In [0]:
%sql
SELECT *
FROM adb_healthcare.gold.hospital_daily_summary
WHERE hospital_id = (
  SELECT hospital_id FROM adb_healthcare.gold.hospital_daily_summary
  GROUP BY hospital_id ORDER BY SUM(total_tests) DESC, hospital_id LIMIT 1)
ORDER BY summary_date
LIMIT 15;

In [0]:
# DBTITLE 1,Return hospital daily summary record count
updated_df = spark.table("adb_healthcare.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))